# Mutual Information — concept 39

Interactive companion to `README.md` and `lesson.tex`.

We will:
1. Recall the definitions of $I(X;Y)$.
2. Implement an empirical plug-in MI estimator (stdlib only).
3. Verify the limiting cases $I=0$ (independent) and $I=H(X)$ ($Y=X$).
4. Sweep the binary symmetric channel and compare empirical MI to
   $1 - H_2(\varepsilon)$.
5. Visualize the curve $\varepsilon \mapsto I(X;Y)$ with an ASCII plot.
6. Discuss the connection to InfoNCE / contrastive learning.


## 1. Definitions (recap)

For discrete $X, Y$ with joint pmf $p(x, y)$ and marginals $p(x), p(y)$,
$$
I(X; Y) = \sum_{x, y} p(x, y) \log_2 \frac{p(x, y)}{p(x)\, p(y)}
       = H(X) - H(X \mid Y).
$$
We use $\log_2$ throughout, so $I$ is in **bits**.

In [ ]:
import math, random
from collections import Counter

def entropy_bits(probs):
    return -sum(p * math.log2(p) for p in probs if p > 0.0)

def mutual_information_bits(samples):
    n = 0
    joint, px, py = Counter(), Counter(), Counter()
    for x, y in samples:
        joint[(x, y)] += 1
        px[x] += 1
        py[y] += 1
        n += 1
    mi = 0.0
    for (x, y), nxy in joint.items():
        pxy = nxy / n
        denom = (px[x] / n) * (py[y] / n)
        mi += pxy * math.log2(pxy / denom)
    return mi

def binary_entropy(eps):
    return entropy_bits([eps, 1.0 - eps])

rng = random.Random(0)
print("helpers ready")


## 2. Limiting cases

- Two independent fair coins: expect $I(X;Y) = 0$.
- $Y = X$ exactly: expect $I(X;Y) = H(X) = 1$ bit.

In [ ]:
N = 200_000

indep = [(rng.randrange(2), rng.randrange(2)) for _ in range(N)]
print(f"independent fair coins: I = {mutual_information_bits(indep):+.5f} bits")

eq = [(b, b) for b in (rng.randrange(2) for _ in range(N))]
print(f"Y = X exactly:          I = {mutual_information_bits(eq):+.5f} bits")


## 3. Binary symmetric channel sweep

Take $X \sim \mathrm{Bernoulli}(1/2)$ and $Y = X \oplus N$ with
$N \sim \mathrm{Bernoulli}(\varepsilon)$. The closed form is
$I(X;Y) = 1 - H_2(\varepsilon)$ bits. We compare against the empirical
plug-in estimate.

In [ ]:
def bsc_samples(n, eps):
    out = []
    for _ in range(n):
        x = rng.randrange(2)
        flip = 1 if rng.random() < eps else 0
        out.append((x, x ^ flip))
    return out

results = []
for eps in [0.0, 0.05, 0.10, 0.25, 0.40, 0.50, 0.75, 0.90, 1.0]:
    analytic = 1.0 - binary_entropy(eps) if 0.0 < eps < 1.0 else 1.0
    emp = mutual_information_bits(bsc_samples(50_000, eps))
    results.append((eps, analytic, emp))
    print(f"eps={eps:.2f}  analytic={analytic:.5f}  empirical={emp:.5f}  |diff|={abs(analytic-emp):.4f}")


## 4. ASCII plot of $\varepsilon \mapsto I(X;Y)$

A quick visual sanity check: the curve should be U-shaped, hitting
$I = 1$ at $\varepsilon = 0$ and $\varepsilon = 1$, and dipping to
$I = 0$ at $\varepsilon = 1/2$.

In [ ]:
width = 50
print("eps   I(X;Y)  |" + "-" * width + "|")
for eps_int in range(0, 21):
    eps = eps_int / 20.0
    I = 1.0 - binary_entropy(eps) if 0.0 < eps < 1.0 else 1.0
    bar = "#" * int(round(I * width))
    print(f"{eps:4.2f}  {I:5.3f}  |{bar:<{width}}|")


## 5. Why this matters: InfoNCE as an MI lower bound

In contrastive learning (CPC, SimCLR, CLIP) we sample $K-1$ negatives
$y_2, \dots, y_K$ alongside the true pair $(x, y_1)$ and minimize
$$
\mathcal{L}_{\mathrm{NCE}} = -\mathbb{E}\!\left[
  \log \frac{f(x, y_1)}{\sum_{k=1}^{K} f(x, y_k)}
\right].
$$
A standard result (van den Oord et al., 2018) gives
$$
I(X; Y) \;\ge\; \log K - \mathcal{L}_{\mathrm{NCE}}.
$$
So pushing the loss down tightens a lower bound on mutual information,
which is why contrastive pretraining produces representations that
preserve task-relevant signal. The same MI lens explains the
**information bottleneck** trade-off and the monotonic-MI guarantees
along the trajectory of an entropic-Lagrangian flow (arXiv:2605.10938).

### Self-check exercises

1. Modify `bsc_samples` to use $X \sim \mathrm{Bernoulli}(p)$ for
   general $p$. Does the symmetric formula $I = 1 - H_2(\varepsilon)$
   still hold? If not, derive the corrected expression.
2. Implement $I(X; Y)$ for a *Z-channel* (asymmetric: only $0 \to 1$
   flips with probability $\varepsilon$). Sweep $\varepsilon$ and
   compare to the BSC curve.
3. Plug-in MI estimators are biased upward at small $n$; sample
   $n \in \{50, 500, 5000, 50000\}$ from two independent fair coins
   and tabulate the bias.